# Abstraction is the blueprint and not the actual object what we can create. 
# we must import abc (Abstract Base Class) from python library
# An abstract method does not have any body defined at the base

In [32]:
# Importing ABC class and abstractmethod from 'abc' library
from abc import ABC, abstractmethod
# Creating an abstract class
class AutomatedAsset(ABC):
    def __init__(self, asset_id: str, environment: str):
        self.asset_id = asset_id
        self.environment = environment
    
    # Defining an abstract method
    @abstractmethod
    def deploy_resource(self):
        pass

    @abstractmethod
    def terminate_resource(self):
        pass

In [33]:
# Let's try to directly create the abstract class by creating an object. It should throw error
print("Trying to create an object of AutomatedAsset class")
try:
    generic_item = AutomatedAsset("dev_item1", "dev")
except Exception as e:
    print(f"🔒 Python Blocked It: {e}")

Trying to create an object of AutomatedAsset class
🔒 Python Blocked It: Can't instantiate abstract class AutomatedAsset without an implementation for abstract methods 'deploy_resource', 'terminate_resource'


In [34]:
# Defining Red Cross and Green Tick to use with our programs
RED_CROSS = "\033[1;31m❌\033[0m"
GREEN_TICK_BOX = "\033[1;32m☑️\033[0m"

In [35]:
# Creating a class to use baseclass 
import ipaddress
print("Creating a class by using class AutomatedAsset by inheriting it")

class LinuxComputeInstance(AutomatedAsset):
    def __init__(self, asset_id: str, environment: str, ip_address: str):
        #Calling the super class constracter 
        super().__init__(asset_id, environment)
        self.__ip_address = None 

        self.ip_address = ip_address
    
    # Getter
    @property
    def ip_address(self):
        return self.__ip_address
    
    # Setter
    @ip_address.setter 
    def ip_address(self, ip_addr):
        try:
            # Checking if the given IP address is in correct format 
            ipaddress.ip_address(ip_addr)
        except TypeError as te:
            raise TypeError(f"{RED_CROSS} IP Address format is not valid : {str(te)}")
        
        # if structure is correct then assign the value to
        self.__ip_address = ip_addr 
    
    def deploy_resource(self, ip_addr:str): 
        # Callng ip_address method inside the deploy_resource
        self.ip_address = ip_addr 
        return(f"Asset-id : {self.asset_id} deployed with IP Address : {self.ip_address}")

    
    def terminate_resource(self):
        return(f"Asset-id : {self.asset_id} destroyed")

Creating a class by using class AutomatedAsset by inheriting it


In [36]:
# Creating GetwayConnection Class which will be using Abstract Class 'AutomatedAsset'

class GatewayConnection(AutomatedAsset):
    def __init__(self, asset_id: str, environment: str, api_tok: str):
        # Calling the super class constractor
        super().__init__(asset_id, environment)
        self.__api_token =None 

        self.api_token = api_tok
    
    # Getter for api_token
    @property
    def api_token(self):
        tok=str(self.__api_token)
        if not tok.startswith("sk-"):
            raise ValueError(f"{RED_CROSS} API Token Not started with 'sk-'")

        try:
            masked_key = tok[:3] + "•" * (len(tok) - 7) + tok[-4:]
            return masked_key
        except Exception as e:
            raise RuntimeError(f"\n {RED_CROSS} Error Details => Error while masking {str(e)}")

    # Setter for api_token
    def api_token(self, new_token: str):
        clean_token = new_token.strip()
        # Check if token is atleast 20 characters long
        if len(clean_token) < 20:
            raise ValueError(f"\n {RED_CROSS} API Token length is not matching with the requirement (min 20 ch long)")
        # Check if new token is starts with 'sk' 
        
        if not clean_token.startswith("sk-"):
            raise ValueError(f"\n {RED_CROSS} API Token Not started with 'sk-'")
        
        self.__api_token = clean_token
    
    # Using parent abstract class methods

    def deploy_resource(self, api_tok:str): 
        # Callng ip_address method inside the deploy_resource
        self.api_token = api_tok
        return(f"Opening secure API channel {self.asset_id} using token prefix : {self.api_token}")


    def terminate_resource(self):
        return(f"Revoking token and sealing API gateway channel {self.asset_id}.")

In [37]:
# Creating Infrastructure. Here we will try to assign a free IP to Linux Object and Will try to generate
# API Token to assign to the object
import secrets
import string

Available_IP = ["10.10.1.10", "10.10.2.40", "10.10.3.50", "10.10.6.10"]
Get_Env = ["prod", "dev", "prod", "test"]
asset_queue = []


# We will generate random API key's starting with 'sk-' and combined with characters and digits
prefix = "sk-"
random_length = max(0, 20 - len(prefix))
pool = string.ascii_letters + string.digits

# We will create 4 objects of LinuxServers and API's
for i in range(len(Available_IP)):
    #print(f"[INFO] Current value is {i}")
    # Creating LinuxComputInstance
    try:
        Lin_Serv = LinuxComputeInstance(f"LinServ-{( i + 1)}", Get_Env[i], Available_IP.pop())
        print(f"{GREEN_TICK_BOX} Linux Server Object Created Successfully")
    except Exception as le:
        print(f"{RED_CROSS} Runtime Error while creating Linux Compute Object\n Error : {str(le)}")
    # Generating random API key
    random_secret = ''.join(secrets.choice(pool) for _ in range(random_length))
    new_api_key = prefix + random_secret
    #print(f"[INFO] Current API Key : {api_key}")
    # Creating API Key Object
    try:
        AP_Key = GatewayConnection(f"API-Tok-{(i + 1)}", Get_Env[i], new_api_key)
        print(f"{GREEN_TICK_BOX} API Key Object created successfully")
    except Exception as ae:
        print(f"{RED_CROSS} Runtime Error while creating API-KEY Object\n Error : {str(ae)}")
    #Adding the objects to asset_queue
    asset_queue.append(Lin_Serv)
    asset_queue.append(AP_Key)

    

☑️ Linux Server Object Created Successfully
☑️ API Key Object created successfully
☑️ Linux Server Object Created Successfully
☑️ API Key Object created successfully
☑️ Linux Server Object Created Successfully
☑️ API Key Object created successfully
☑️ Linux Server Object Created Successfully
☑️ API Key Object created successfully


In [38]:
# Displaying the Objects
for asset in asset_queue:
    if "Lin" in asset.asset_id:
        print(f"Asset ID : {asset.asset_id} having IP of : {asset.ip_address}")
    elif "API" in asset.asset_id:
        print(f"Asset ID : {asset.asset_id} having IP of : {asset.api_token}\n")


Asset ID : LinServ-1 having IP of : 10.10.6.10
Asset ID : API-Tok-1 having IP of : sk-aqyggcGMgFvQT0ovJ

Asset ID : LinServ-2 having IP of : 10.10.3.50
Asset ID : API-Tok-2 having IP of : sk-q2SueX6DRjtf20O7a

Asset ID : LinServ-3 having IP of : 10.10.2.40
Asset ID : API-Tok-3 having IP of : sk-P9yfy3OvCaUBLMYdm

Asset ID : LinServ-4 having IP of : 10.10.1.10
Asset ID : API-Tok-4 having IP of : sk-hitfi478I8TgS4Q8Y



In [39]:
# Changing IP Address of a Specific Linux Server
def IPChange(Serv_ID, New_IP):
    print("This function block can help changing the IP adderss of a given Linux Server\n\n")
    result = ""

    asset_found = False
    for serv in asset_queue:
        if serv.asset_id == Serv_ID:
            asset_found = True
            try:
                result  = serv.deploy_resource(New_IP)
            except Exception as e:
                result = f"{RED_CROSS} Error occured {str(e)}"
            break


    if not asset_found:
        result=(f"{RED_CROSS} Mentioned Server not found in the asset list")

    return result

In [28]:
Serv_ID = input("Enter the Server-ID :").strip()
New_IP = input("Enter the New IP Address :").strip()
result = IPChange(Serv_ID, New_IP)
print(result)

This function block can help changing the IP adderss of a given Linux Server


Asset-id : LinServ-2 deployed with IP Address : 10.10.3.55


In [40]:
# Deleting all objects

for asset in asset_queue:
    response=asset.terminate_resource()
    print(response)

Asset-id : LinServ-1 destroyed
Revoking token and sealing API gateway channel API-Tok-1.
Asset-id : LinServ-2 destroyed
Revoking token and sealing API gateway channel API-Tok-2.
Asset-id : LinServ-3 destroyed
Revoking token and sealing API gateway channel API-Tok-3.
Asset-id : LinServ-4 destroyed
Revoking token and sealing API gateway channel API-Tok-4.
